<a href="https://colab.research.google.com/github/dilnaznur/Prime/blob/main/Dyslex2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ЯЧЕЙКА 1: Установка библиотек
print("📦 Устанавливаем необходимые библиотеки...")

# Основные библиотеки уже есть в Colab, проверим версии
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import scipy

print("✅ NumPy:", np.__version__)
print("✅ Pandas:", pd.__version__)
print("✅ Scikit-learn:", sklearn.__version__)
print("✅ SciPy:", scipy.__version__)

# Установим дополнительные (если нужны)
!pip install -q tqdm

print("\n✅ Все библиотеки установлены!")

📦 Устанавливаем необходимые библиотеки...
✅ NumPy: 2.0.2
✅ Pandas: 2.2.2
✅ Scikit-learn: 1.6.1
✅ SciPy: 1.16.3

✅ Все библиотеки установлены!


In [ ]:
# ЯЧЕЙКА 2: Загрузка датасета ETDD70 через API
import os
import requests
import zipfile
from tqdm import tqdm

print("📥 ЗАГРУЗКА ДАТАСЕТА ETDD70 (через Zenodo API)")
print("="*60)

# Создаем папку для данных
os.makedirs('data', exist_ok=True)

# Zenodo record ID
record_id = "13332134"

# Получаем информацию о файлах через API
print("🔍 Получаем информацию о датасете...")
api_url = f"https://zenodo.org/api/records/{record_id}"

try:
    response = requests.get(api_url)
    response.raise_for_status()
    record_data = response.json()

    # Находим файлы
    files = record_data['files']

    print(f"✅ Найдено файлов: {len(files)}")

    # Ищем ZIP файл
    zip_file = None
    for file in files:
        if file['key'].endswith('.zip'):
            zip_file = file
            break

    if zip_file:
        print(f"\n📦 Файл для скачивания: {zip_file['key']}")
        print(f"   Размер: {zip_file['size'] / 1024 / 1024:.2f} MB")

        # Скачиваем
        download_url = zip_file['links']['self']
        zip_path = f"data/{zip_file['key']}"

        if not os.path.exists(zip_path):
            print("\n⬇️  Начинаем загрузку...")

            response = requests.get(download_url, stream=True)
            total_size = int(response.headers.get('content-length', 0))

            with open(zip_path, 'wb') as f, tqdm(
                desc=zip_file['key'],
                total=total_size,
                unit='B',
                unit_scale=True,
                unit_divisor=1024,
            ) as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))

            print("✅ Загрузка завершена!")
        else:
            print("✅ Файл уже скачан!")

        # Распаковываем
        print("\n📂 Распаковываем архив...")
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                # Показываем прогресс распаковки
                file_list = zip_ref.namelist()
                print(f"   Файлов в архиве: {len(file_list)}")

                for file in tqdm(file_list, desc="Распаковка"):
                    zip_ref.extract(file, 'data/')

            print("✅ Данные готовы к использованию!")

        except Exception as e:
            print(f"❌ Ошибка распаковки: {e}")

    else:
        print("❌ ZIP файл не найден в записи")
        print("\n📋 Доступные файлы:")
        for file in files:
            print(f"  - {file['key']} ({file['size'] / 1024 / 1024:.2f} MB)")

except requests.exceptions.RequestException as e:
    print(f"❌ Ошибка подключения к Zenodo API: {e}")
    print("\n💡 АЛЬТЕРНАТИВНЫЙ СПОСОБ:")
    print("1. Откройте в браузере: https://zenodo.org/records/13332134")
    print("2. Нажмите кнопку 'Download' возле файла ETDD70.zip")
    print("3. После скачивания загрузите файл в Colab:")
    print("   - Нажмите 📁 (Files) в левой панели")
    print("   - Нажмите ⬆️ (Upload)")
    print("   - Выберите скачанный ETDD70.zip")
    print("   - Переместите в папку data/")

except Exception as e:
    print(f"❌ Неожиданная ошибка: {e}")

print("="*60)

📥 ЗАГРУЗКА ДАТАСЕТА ETDD70 (через Zenodo API)
🔍 Получаем информацию о датасете...
✅ Найдено файлов: 6

📦 Файл для скачивания: stimuli.zip
   Размер: 0.20 MB

⬇️  Начинаем загрузку...


stimuli.zip: 100%|██████████| 208k/208k [00:00<00:00, 367kB/s]


✅ Загрузка завершена!

📂 Распаковываем архив...
   Файлов в архиве: 8


Распаковка: 100%|██████████| 8/8 [00:00<00:00, 895.05it/s]

✅ Данные готовы к использованию!


In [ ]:
# ЯЧЕЙКА 3: Скачиваем ВСЕ файлы датасета ETDD70
import os
import requests
import zipfile
from tqdm import tqdm

print("📥 ЗАГРУЗКА ПОЛНОГО ДАТАСЕТА ETDD70")
print("="*60)

# Создаем папку
os.makedirs('data', exist_ok=True)

# Zenodo record ID
record_id = "13332134"

# Получаем информацию о файлах
print("🔍 Получаем список файлов...")
api_url = f"https://zenodo.org/api/records/{record_id}"

try:
    response = requests.get(api_url)
    response.raise_for_status()
    record_data = response.json()

    files = record_data['files']

    print(f"\n📋 Доступные файлы в датасете:")
    for idx, file in enumerate(files, 1):
        size_mb = file['size'] / 1024 / 1024
        print(f"  {idx}. {file['key']} ({size_mb:.2f} MB)")

    print("\n⬇️  Начинаем загрузку всех файлов...")
    print("="*60)

    for file in files:
        file_name = file['key']
        file_path = f"data/{file_name}"

        # Проверяем, не скачан ли уже файл
        if os.path.exists(file_path):
            print(f"\n✅ {file_name} - уже скачан, пропускаем")
            continue

        print(f"\n📦 Скачиваем: {file_name}")

        # Скачиваем
        download_url = file['links']['self']
        response = requests.get(download_url, stream=True)
        total_size = int(response.headers.get('content-length', 0))

        with open(file_path, 'wb') as f, tqdm(
            desc=file_name,
            total=total_size,
            unit='B',
            unit_scale=True,
            unit_divisor=1024,
        ) as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

        print(f"✅ {file_name} загружен!")

        # Если это ZIP, распаковываем
        if file_name.endswith('.zip'):
            print(f"   📂 Распаковываем {file_name}...")
            try:
                with zipfile.ZipFile(file_path, 'r') as zip_ref:
                    extract_path = f"data/{file_name.replace('.zip', '')}"
                    os.makedirs(extract_path, exist_ok=True)

                    file_list = zip_ref.namelist()
                    for zipped_file in tqdm(file_list, desc="   Распаковка"):
                        zip_ref.extract(zipped_file, extract_path)

                print(f"   ✅ {file_name} распакован!")
            except Exception as e:
                print(f"   ⚠️  Ошибка распаковки: {e}")

    print("\n" + "="*60)
    print("✅ ВСЕ ФАЙЛЫ ЗАГРУЖЕНЫ И РАСПАКОВАНЫ!")
    print("="*60)

    # Показываем структуру
    print("\n📁 Структура данных:")
    for root, dirs, files in os.walk('data'):
        level = root.replace('data', '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        for file in files[:5]:
            print(f'{subindent}{file}')
        if len(files) > 5:
            print(f'{subindent}... и еще {len(files)-5} файлов')

except Exception as e:
    print(f"❌ Ошибка: {e}")

print("="*60)

📥 ЗАГРУЗКА ПОЛНОГО ДАТАСЕТА ETDD70
🔍 Получаем список файлов...

📋 Доступные файлы в датасете:
  1. stimuli.zip (0.20 MB)
  2. README.md (0.00 MB)
  3. fixation_images.zip (21.07 MB)
  4. rois.zip (0.01 MB)
  5. data.zip (127.55 MB)
  6. dyslexia_class_label.csv (0.00 MB)

⬇️  Начинаем загрузку всех файлов...

✅ stimuli.zip - уже скачан, пропускаем

📦 Скачиваем: README.md


README.md: 100%|██████████| 3.01k/3.01k [00:00<00:00, 7.83MB/s]


✅ README.md загружен!

📦 Скачиваем: fixation_images.zip


fixation_images.zip: 100%|██████████| 21.1M/21.1M [00:27<00:00, 800kB/s]


✅ fixation_images.zip загружен!
   📂 Распаковываем fixation_images.zip...


...

✅ data.zip загружен!
   📂 Распаковываем data.zip...


   Распаковка: 100%|██████████| 1682/1682 [00:05<00:00, 312.61it/s]


   ✅ data.zip распакован!

📦 Скачиваем: dyslexia_class_label.csv


dyslexia_class_label.csv: 1.33kB [00:00, 4.00MB/s]

✅ dyslexia_class_label.csv загружен!

✅ ВСЕ ФАЙЛЫ ЗАГРУЖЕНЫ И РАСПАКОВАНЫ!

📁 Структура данных:
data/
  data.zip
  stimuli.zip
  fixation_images.zip
  README.md
  dyslexia_class_label.csv
  ... и еще 1 файлов
  __MACOSX/
    ._images
    images/
      ._s7_stimuli_t4.jpg
      ._s7_stimuli_t1.jpg
      ._s7_stimuli_t5.jpg
  images/
    s7_stimuli_t5.jpg
    s7_stimuli_t1.jpg
    s7_stimuli_t4.jpg
  fixation_images/
    fixation_images/
      PseudoText/
        Subject_1421_PseudoText.png
        Subject_1109_PseudoText.png
        Subject_1993_PseudoText.png
        Subject_1903_PseudoText.png
        Subject_1318_PseudoText.png
        ... и еще 65 файлов
      Syllables/
        Subject_1459_Syllables.png
        Subject_1073_Syllables.png
        Subject_1349_Syllables.png
        Subject_1913_Syllables.png
        Subject_1398_Syllables.png
        ... и еще 65 файлов
      MeaningfulText/
        Subject_1258_MeaningfulText.png
        Subject_1417_MeaningfulText.png
        Subj

In [ ]:
# ЯЧЕЙКА 4: Фильтруем реальные файлы (без служебных)
import os
import pandas as pd

print("🔍 ФИЛЬТРАЦИЯ РЕАЛЬНЫХ ФАЙЛОВ")
print("="*60)

# Ищем все CSV файлы, НО пропускаем служебные (._*)
all_csv_files = []
for root, dirs, files in os.walk('data/data'):
    for file in files:
        # Пропускаем служебные файлы macOS
        if file.endswith('.csv') and not file.startswith('._'):
            full_path = os.path.join(root, file)
            all_csv_files.append(full_path)

print(f"✅ Найдено реальных CSV файлов: {len(all_csv_files)}")

if len(all_csv_files) > 0:
    # Анализируем типы файлов
    fixation_files = [f for f in all_csv_files if 'fixation' in f.lower()]
    raw_files = [f for f in all_csv_files if 'raw' in f.lower()]
    saccade_files = [f for f in all_csv_files if 'saccade' in f.lower()]
    metrics_files = [f for f in all_csv_files if 'metric' in f.lower()]

    print(f"\n📊 Типы файлов:")
    print(f"  • Фиксации: {len(fixation_files)}")
    print(f"  • Сырые данные: {len(raw_files)}")
    print(f"  • Саккады: {len(saccade_files)}")
    print(f"  • Метрики: {len(metrics_files)}")

    # Показываем примеры
    print(f"\n📋 Примеры РЕАЛЬНЫХ файлов:")
    for f in all_csv_files[:10]:
        filename = f.split('/')[-1]
        size = os.path.getsize(f) / 1024  # KB
        print(f"  • {filename} ({size:.1f} KB)")

    # Загружаем файл с фиксациями (самые важные!)
    print(f"\n" + "="*60)
    print(f"📊 АНАЛИЗ ФАЙЛА С ФИКСАЦИЯМИ")
    print(f"="*60)

    if len(fixation_files) > 0:
        example_file = fixation_files[0]
        filename = example_file.split('/')[-1]
        print(f"\nОткрываем: {filename}")

        try:
            df = pd.read_csv(example_file)

            print(f"\n✅ Файл загружен!")
            print(f"Размер: {df.shape[0]} строк × {df.shape[1]} колонок")

            print(f"\n📋 Колонки в файле:")
            for i, col in enumerate(df.columns, 1):
                print(f"  {i}. {col}")

            print(f"\n📈 Первые 10 строк:")
            print(df.head(10))

            print(f"\n📊 Статистика:")
            print(df.describe())

            print(f"\n" + "="*60)
            print(f"🔍 ЧТО ОЗНАЧАЮТ КОЛОНКИ:")
            print(f"="*60)

            # Анализируем каждую колонку
            for col in df.columns:
                col_lower = col.lower()

                if 'fixation_x' in col_lower or col_lower == 'x':
                    print(f"  ✓ {col}: X координата фиксации (пиксели)")
                elif 'fixation_y' in col_lower or col_lower == 'y':
                    print(f"  ✓ {col}: Y координата фиксации (пиксели)")
                elif 'duration' in col_lower:
                    print(f"  ✓ {col}: Длительность фиксации (миллисекунды)")
                elif 'start' in col_lower:
                    print(f"  ✓ {col}: Время начала фиксации")
                elif 'end' in col_lower:
                    print(f"  ✓ {col}: Время окончания фиксации")
                elif 'dispersion' in col_lower:
                    print(f"  ✓ {col}: Разброс взгляда (стабильность)")
                elif 'pupil' in col_lower:
                    print(f"  ✓ {col}: Размер зрачка")
                elif 'word' in col_lower:
                    print(f"  ✓ {col}: Номер/ID слова")
                elif 'line' in col_lower:
                    print(f"  ✓ {col}: Номер строки")
                elif 'roi' in col_lower:
                    print(f"  ✓ {col}: Регион интереса (ROI)")
                elif 'char' in col_lower:
                    print(f"  ✓ {col}: Символ/позиция в тексте")

            # Сохраняем для дальнейшего использования
            print(f"\n💾 Информация о структуре сохранена!")

        except Exception as e:
            print(f"❌ Ошибка чтения: {e}")
    else:
        print("❌ Файлы с фиксациями не найдены")

else:
    print("❌ Реальные CSV файлы не найдены!")

print("="*60)

🔍 ФИЛЬТРАЦИЯ РЕАЛЬНЫХ ФАЙЛОВ
✅ Найдено реальных CSV файлов: 840

📊 Типы файлов:
  • Фиксации: 210
  • Сырые данные: 210
  • Саккады: 210
  • Метрики: 210

📋 Примеры РЕАЛЬНЫХ файлов:
  • Subject_1300_T5_Pseudo_Text_fixations.csv (89.7 KB)
  • Subject_1322_T1_Syllables_metrics.csv (35.0 KB)
  • Subject_1145_T1_Syllables_saccades.csv (42.6 KB)
  • Subject_1582_T5_Pseudo_Text_fixations.csv (66.0 KB)
  • Subject_1993_T1_Syllables_metrics.csv (36.8 KB)
  • Subject_1003_T4_Meaningful_Text_fixations.csv (27.0 KB)
  • Subject_1913_T1_Syllables_metrics.csv (36.7 KB)
  • Subject_1187_T1_Syllables_raw.csv (1160.5 KB)
  • Subject_1189_T5_Pseudo_Text_metrics.csv (39.5 KB)
  • Subject_1760_T5_Pseudo_Text_fixations.csv (47.4 KB)

📊 АНАЛИЗ ФАЙЛА С ФИКСАЦИЯМИ

Открываем: Subject_1300_T5_Pseudo_Text_fixations.csv

✅ Файл загружен!
Размер: 474 строк × 17 колонок

📋 Колонки в файле:
  1. id
  2. task
  3. sid
  4. eye
  5. stimfile
  6. trialid
  7. start_ms
  8. end_ms
  9. duration_ms
  10. fix_x
  11. f

In [ ]:
"""
📊 ЗАГРУЗКА И ОБРАБОТКА РЕАЛЬНЫХ ДАННЫХ ETDD70
Этот скрипт работает с настоящей структурой датасета ETDD70
"""

import pandas as pd
import numpy as np
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("="*90)
print("📊 ЗАГРУЗКА РЕАЛЬНЫХ ДАННЫХ ETDD70")
print("="*90)

# ============================================================================
# ШАГ 1: ЗАГРУЗКА МЕТОК КЛАССОВ
# ============================================================================

print("\n🏷️  ШАГ 1: Загрузка меток классов (dyslexia_class_label.csv)")
print("-"*90)

# Путь к вашим данным (измените если нужно)
DATA_DIR = "data"  # Папка где у вас распакованы данные

try:
    labels_df = pd.read_csv(f"{DATA_DIR}/dyslexia_class_label.csv")
    print(f"✅ Загружено меток: {len(labels_df)}")
    print(f"\nКолонки: {list(labels_df.columns)}")
    print(f"\nПервые 5 строк:")
    print(labels_df.head())

    # Подсчёт классов
    if 'label' in labels_df.columns or 'class' in labels_df.columns:
        label_col = 'label' if 'label' in labels_df.columns else 'class'
        print(f"\n📊 Распределение классов:")
        print(labels_df[label_col].value_counts())
except FileNotFoundError:
    print("❌ Файл dyslexia_class_label.csv не найден!")
    print(f"   Проверьте путь: {DATA_DIR}/dyslexia_class_label.csv")
    exit(1)

# ============================================================================
# ШАГ 2: СКАНИРОВАНИЕ ФАЙЛОВ С ДАННЫМИ
# ============================================================================

print("\n" + "="*90)
print("📁 ШАГ 2: Сканирование файлов с данными")
print("-"*90)

data_path = Path(f"{DATA_DIR}/data/data")  # data/data/data по вашей структуре

if not data_path.exists():
    print(f"❌ Путь не найден: {data_path}")
    print("\n💡 Пожалуйста, укажите правильный путь к папке с CSV файлами")
    print("   Структура должна быть: data/data/data/*.csv")
    exit(1)

# Найти все CSV файлы
all_files = list(data_path.glob("*.csv"))
print(f"✅ Найдено файлов: {len(all_files)}")

# Фильтруем реальные файлы (без __MACOSX)
real_files = [f for f in all_files if '__MACOSX' not in str(f)]
print(f"✅ Реальных файлов (без служебных): {len(real_files)}")

# Группируем по типам
fixation_files = [f for f in real_files if 'fixations.csv' in f.name]
metrics_files = [f for f in real_files if 'metrics.csv' in f.name]
saccades_files = [f for f in real_files if 'saccades.csv' in f.name]

print(f"\n📊 Типы файлов:")
print(f"   • Фиксации (fixations): {len(fixation_files)}")
print(f"   • Метрики (metrics): {len(metrics_files)}")
print(f"   • Саккады (saccades): {len(saccades_files)}")

# ============================================================================
# ШАГ 3: ИЗВЛЕЧЕНИЕ ПРИЗНАКОВ ИЗ FIXATIONS
# ============================================================================

print("\n" + "="*90)
print("🔍 ШАГ 3: Извлечение признаков из файлов fixations")
print("-"*90)

def extract_subject_id(filename):
    """Извлекает ID субъекта из имени файла"""
    # Формат: Subject_1169_T1_Syllables_fixations.csv
    parts = filename.split('_')
    return int(parts[1])  # Возвращаем числовой ID

def extract_task(filename):
    """Извлекает тип задания"""
    if 'T1' in filename:
        return 'T1_Syllables'
    elif 'T4' in filename:
        return 'T4_Meaningful'
    elif 'T5' in filename:
        return 'T5_Pseudo'
    return 'Unknown'

def calculate_tvi_features(fixations_df):
    """
    Рассчитывает TVI компоненты из данных фиксаций
    """
    if len(fixations_df) < 2:
        return None

    durations = fixations_df['duration_ms'].values

    # Базовые признаки
    features = {
        'num_fixations': len(fixations_df),
        'mean_fixation_duration': durations.mean(),
        'median_fixation_duration': np.median(durations),
        'total_reading_time': fixations_df['end_ms'].max() - fixations_df['start_ms'].min(),
        'mean_fixation_x': fixations_df['fix_x'].mean() if 'fix_x' in fixations_df else 0,
        'std_fixation_y': fixations_df['fix_y'].std() if 'fix_y' in fixations_df else 0,
    }

    # TVI КОМПОНЕНТЫ

    # 1. CV fixation duration (коэффициент вариации)
    features['cv_fixation_duration'] = durations.std() / (durations.mean() + 1e-6)

    # 2. Entropy fixation duration
    hist, _ = np.histogram(durations, bins=10)
    prob = hist / (hist.sum() + 1e-6)
    prob = prob[prob > 0]
    features['entropy_fixation_duration'] = -np.sum(prob * np.log2(prob + 1e-6))

    # 3. CV inter-fixation intervals
    if len(fixations_df) > 1:
        intervals = np.diff(fixations_df['start_ms'].values)
        features['cv_inter_fixation_intervals'] = intervals.std() / (intervals.mean() + 1e-6)
    else:
        features['cv_inter_fixation_intervals'] = 0

    # 4. Speed irregularity (вариативность скорости)
    if 'fix_x' in fixations_df and len(fixations_df) > 1:
        distances = np.sqrt(np.diff(fixations_df['fix_x'].values)**2 +
                          np.diff(fixations_df['fix_y'].values)**2)
        speeds = distances / (durations[:-1] + 1e-6)
        features['speed_irregularity'] = speeds.std() / (speeds.mean() + 1e-6)
    else:
        features['speed_irregularity'] = 0

    # 5. Autocorrelation
    if len(durations) > 2:
        durations_norm = (durations - durations.mean()) / (durations.std() + 1e-6)
        autocorr = np.correlate(durations_norm[:-1], durations_norm[1:], mode='valid')
        features['autocorrelation'] = autocorr[0] / len(durations_norm) if len(autocorr) > 0 else 0
    else:
        features['autocorrelation'] = 0

    # 6. Meta-variability (вариативность вариативности)
    if len(durations) > 10:
        window_size = len(durations) // 3
        window_stds = []
        for i in range(len(durations) - window_size):
            window = durations[i:i+window_size]
            window_stds.append(window.std())
        features['meta_variability'] = np.array(window_stds).std() / (np.array(window_stds).mean() + 1e-6)
    else:
        features['meta_variability'] = 0

    # 7. TVI SCORE (взвешенная комбинация)
    features['tvi_score'] = (
        0.20 * features['cv_fixation_duration'] +
        0.18 * min(features['entropy_fixation_duration'] / 2.5, 1.0) +  # normalize
        0.17 * features['cv_inter_fixation_intervals'] +
        0.20 * features['speed_irregularity'] +
        0.12 * (1 - features['autocorrelation']) +
        0.13 * features['meta_variability']
    )

    return features

# Обрабатываем все fixation файлы
all_features = []

print("\n⏳ Обработка файлов...")
print("(это займёт 1-2 минуты)")

for i, file_path in enumerate(fixation_files, 1):
    if i % 50 == 0:
        print(f"   Обработано: {i}/{len(fixation_files)}...")

    try:
        # Загружаем данные
        df = pd.read_csv(file_path)

        # Извлекаем признаки
        features = calculate_tvi_features(df)

        if features is not None:
            # Добавляем метаданные
            features['subject_id'] = extract_subject_id(file_path.name)
            features['task'] = extract_task(file_path.name)
            features['filename'] = file_path.name

            all_features.append(features)

    except Exception as e:
        print(f"⚠️  Ошибка при обработке {file_path.name}: {e}")
        continue

print(f"\n✅ Обработано успешно: {len(all_features)} записей")

# ============================================================================
# ШАГ 4: СОЗДАНИЕ ИТОГОВОГО ДАТАФРЕЙМА
# ============================================================================

print("\n" + "="*90)
print("📊 ШАГ 4: Создание итогового датафрейма")
print("-"*90)

features_df = pd.DataFrame(all_features)

print(f"✅ Создан датафрейм: {len(features_df)} записей")
print(f"   Уникальных субъектов: {features_df['subject_id'].nunique()}")
print(f"   Признаков: {len(features_df.columns)}")

# ============================================================================
# ШАГ 5: ДОБАВЛЕНИЕ МЕТОК КЛАССОВ
# ============================================================================

print("\n" + "="*90)
print("🏷️  ШАГ 5: Добавление меток классов (dyslexia/control)")
print("-"*90)

# Нужно сопоставить subject_id с метками
# Предполагается, что в dyslexia_class_label.csv есть колонка 'subject' или 'sid'

if 'subject' in labels_df.columns:
    label_col_name = 'subject'
elif 'sid' in labels_df.columns:
    label_col_name = 'sid'
elif 'subject_id' in labels_df.columns:
    label_col_name = 'subject_id'
else:
    # Берём первую числовую колонку как ID
    label_col_name = labels_df.columns[0]

print(f"Используем колонку '{label_col_name}' для сопоставления")

# Merge
features_df = features_df.merge(
    labels_df[[label_col_name, label_col]].rename(columns={label_col_name: 'subject_id'}),
    on='subject_id',
    how='left'
)

# Переименуем label_col в 'label' если нужно
if label_col != 'label':
    features_df = features_df.rename(columns={label_col: 'label'})

print(f"✅ Метки добавлены")
print(f"\n📊 Распределение по классам:")
print(features_df['label'].value_counts())

# Проверяем пропуски
missing_labels = features_df['label'].isna().sum()
if missing_labels > 0:
    print(f"\n⚠️  Внимание: {missing_labels} записей без меток!")
    print(f"   Удаляем их...")
    features_df = features_df.dropna(subset=['label'])

# ============================================================================
# ШАГ 6: СОХРАНЕНИЕ
# ============================================================================

print("\n" + "="*90)
print("💾 ШАГ 6: Сохранение обработанных данных")
print("-"*90)

output_file = "extracted_features_real_etdd70.csv"
features_df.to_csv(output_file, index=False)

print(f"✅ Данные сохранены: {output_file}")
print(f"\n📊 Итоговая статистика:")
print(f"   Записей: {len(features_df)}")
print(f"   Субъектов: {features_df['subject_id'].nunique()}")
print(f"   Признаков: {len([c for c in features_df.columns if c not in ['subject_id', 'task', 'filename', 'label']])}")

print(f"\n📋 Колонки:")
for col in features_df.columns:
    print(f"   • {col}")

print(f"\n📊 TVI score по классам:")
for label_val in features_df['label'].unique():
    subset = features_df[features_df['label'] == label_val]
    print(f"   Класс {label_val}: TVI = {subset['tvi_score'].mean():.3f} ± {subset['tvi_score'].std():.3f}")

print("\n" + "="*90)
print("✅ ГОТОВО! Теперь можете запускать эксперименты:")
print("   python tvi_experiments_no_network.py")
print("="*90)

📊 ЗАГРУЗКА РЕАЛЬНЫХ ДАННЫХ ETDD70

🏷️  ШАГ 1: Загрузка меток классов (dyslexia_class_label.csv)
------------------------------------------------------------------------------------------
✅ Загружено меток: 70

Колонки: ['subject_id', 'class_id', 'label']

Первые 5 строк:
   subject_id  class_id         label
0        1003         0  non-dyslexic
1        1016         0  non-dyslexic
2        1019         0  non-dyslexic
3        1033         0  non-dyslexic
4        1040         0  non-dyslexic

📊 Распределение классов:
label
non-dyslexic    35
dyslexic        35
Name: count, dtype: int64

📁 ШАГ 2: Сканирование файлов с данными
------------------------------------------------------------------------------------------
✅ Найдено файлов: 840
✅ Реальных файлов (без служебных): 840

📊 Типы файлов:
   • Фиксации (fixations): 210
   • Метрики (metrics): 210
   • Саккады (saccades): 210

🔍 ШАГ 3: Извлечение признаков из файлов fixations
---------------------------------------------------------

In [ ]:
"""
🏆 TVI-READ: Optimal TVI Framework
═══════════════════════════════════════════════════════════════════

BEST PERFORMING MODEL: 85.71% accuracy

Key Innovation:
- Ablation-guided component selection
- Removes harmful TVI components
- Weighted TVI scoring
- Optimized Random Forest (deeper trees)

Author: [Your Name]
Date: January 2026
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# ============================================================================
# CONFIGURATION
# ============================================================================

# Baseline features
BASELINE_FEATURES = [
    'num_fixations',
    'mean_fixation_duration',
    'median_fixation_duration',
    'total_reading_time',
    'mean_fixation_x',
    'std_fixation_y'
]

# OPTIMAL TVI components (ablation-guided selection)
# These components were identified through ablation study as beneficial
OPTIMAL_TVI_COMPONENTS = [
    'entropy_fixation_duration',      # Critical: -2.86% when removed
    'autocorrelation',                # Critical: -2.86% when removed
    'cv_inter_fixation_intervals',    # Important: -1.43% when removed
    'meta_variability'                # Neutral: 0% when removed but kept
]

# Removed harmful components (identified by ablation study):
# - cv_fixation_duration       (+1.43% accuracy when removed)
# - speed_irregularity         (+1.43% accuracy when removed)

# Component weights for weighted TVI score (based on importance)
TVI_WEIGHTS = {
    'entropy_fixation_duration': 2.86,
    'autocorrelation': 2.86,
    'cv_inter_fixation_intervals': 1.43,
    'meta_variability': 0.50
}

# Best model parameters (found through experimentation)
MODEL_PARAMS = {
    'n_estimators': 200,
    'max_depth': 15,        # Deeper trees performed best
    'random_state': 42,
    'n_jobs': -1
}

# Cross-validation configuration
CV_FOLDS = 10
CV_SHUFFLE = True
CV_RANDOM_STATE = 42

# ============================================================================
# DATA LOADING
# ============================================================================

def load_data(filepath='extracted_features_real_etdd70.csv'):
    """
    Load eye-tracking features from CSV file.

    Parameters:
    -----------
    filepath : str
        Path to the CSV file containing extracted features

    Returns:
    --------
    pd.DataFrame
        DataFrame with features and labels
    """
    print("📊 Loading data...")

    # Load data
    df = pd.read_csv(filepath)

    # Convert labels to binary
    if df['label'].dtype == 'object':
        label_mapping = {
            'dyslexic': 1,
            'non-dyslexic': 0,
            'dyslexia': 1,
            'control': 0
        }
        df['label'] = df['label'].map(label_mapping)

    print(f"✅ Loaded {len(df)} recordings from {df['subject_id'].nunique()} subjects")

    return df

# ============================================================================
# FEATURE ENGINEERING
# ============================================================================

def create_optimal_tvi(df, baseline_features, tvi_components, tvi_weights):
    """
    Create Optimal TVI features with weighted scoring.

    Parameters:
    -----------
    df : pd.DataFrame
        Raw features DataFrame
    baseline_features : list
        List of baseline feature names
    tvi_components : list
        List of optimal TVI component names
    tvi_weights : dict
        Weights for each TVI component

    Returns:
    --------
    pd.DataFrame
        Aggregated features with TVI scores
    """
    print("\n🔬 Creating Optimal TVI features...")

    # Check available features
    available_baseline = [f for f in baseline_features if f in df.columns]
    available_tvi = [f for f in tvi_components if f in df.columns]

    print(f"   Available features: {len(available_baseline)} baseline + {len(available_tvi)} TVI")

    # Aggregate by subject (mean across tasks)
    all_features = available_baseline + available_tvi
    agg_dict = {f: 'mean' for f in all_features}
    agg_dict['label'] = 'first'

    agg_df = df.groupby('subject_id').agg(agg_dict).reset_index()

    # Create regular TVI score (unweighted mean)
    if available_tvi:
        agg_df['tvi_score'] = agg_df[available_tvi].mean(axis=1)

    # Create weighted TVI score
    if available_tvi:
        weighted_sum = sum(
            agg_df[comp] * tvi_weights.get(comp, 1.0)
            for comp in available_tvi
        )
        weight_total = sum(tvi_weights.get(comp, 1.0) for comp in available_tvi)
        agg_df['weighted_tvi_score'] = weighted_sum / weight_total

        print("   ✅ Created regular and weighted TVI scores")

    return agg_df

# ============================================================================
# MODEL TRAINING & EVALUATION
# ============================================================================

def train_and_evaluate(X, y, model_params, cv_folds=10):
    """
    Train and evaluate model using cross-validation.

    Parameters:
    -----------
    X : np.ndarray
        Feature matrix
    y : np.ndarray
        Target labels
    model_params : dict
        Random Forest parameters
    cv_folds : int
        Number of cross-validation folds

    Returns:
    --------
    dict
        Dictionary with evaluation results
    """
    print(f"\n🤖 Training model with {cv_folds}-fold cross-validation...")

    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Cross-validation
    skf = StratifiedKFold(
        n_splits=cv_folds,
        shuffle=CV_SHUFFLE,
        random_state=CV_RANDOM_STATE
    )

    # Storage for results
    fold_scores = []
    all_predictions = []
    all_true_labels = []
    all_probabilities = []

    # Train and evaluate on each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y), 1):
        # Create and train model
        model = RandomForestClassifier(**model_params)
        model.fit(X_scaled[train_idx], y[train_idx])

        # Predict on test set
        y_pred = model.predict(X_scaled[test_idx])
        y_proba = model.predict_proba(X_scaled[test_idx])

        # Calculate accuracy
        fold_acc = accuracy_score(y[test_idx], y_pred)
        fold_scores.append(fold_acc)

        # Store predictions
        all_predictions.extend(y_pred)
        all_true_labels.extend(y[test_idx])
        all_probabilities.extend(np.max(y_proba, axis=1))

    # Convert to arrays
    all_predictions = np.array(all_predictions)
    all_true_labels = np.array(all_true_labels)
    all_probabilities = np.array(all_probabilities)

    # Calculate overall metrics
    overall_accuracy = np.mean(fold_scores)
    std_accuracy = np.std(fold_scores)

    # Confusion matrix
    cm = confusion_matrix(all_true_labels, all_predictions)
    tn, fp, fn, tp = cm.ravel()

    # Clinical metrics
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0

    # Confidence-based analysis
    high_conf_mask = all_probabilities > 0.90
    high_conf_accuracy = accuracy_score(
        all_true_labels[high_conf_mask],
        all_predictions[high_conf_mask]
    ) if high_conf_mask.sum() > 0 else 0
    high_conf_percentage = (high_conf_mask.sum() / len(all_probabilities)) * 100

    # Print results
    print(f"\n📊 Results:")
    print(f"   Overall Accuracy:     {overall_accuracy:.4f} ± {std_accuracy:.4f} ({overall_accuracy*100:.2f}%)")
    print(f"   High-Confidence:      {high_conf_accuracy:.4f} ({high_conf_accuracy*100:.1f}%) on {high_conf_percentage:.1f}% samples")
    print(f"\n   Clinical Metrics:")
    print(f"   Sensitivity:          {sensitivity:.4f} ({sensitivity*100:.1f}%)")
    print(f"   Specificity:          {specificity:.4f} ({specificity*100:.1f}%)")
    print(f"   PPV:                  {ppv:.4f} ({ppv*100:.1f}%)")
    print(f"   NPV:                  {npv:.4f} ({npv*100:.1f}%)")

    # Return results
    results = {
        'accuracy_mean': overall_accuracy,
        'accuracy_std': std_accuracy,
        'fold_scores': fold_scores,
        'confusion_matrix': cm,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'ppv': ppv,
        'npv': npv,
        'high_conf_accuracy': high_conf_accuracy,
        'high_conf_percentage': high_conf_percentage,
        'predictions': all_predictions,
        'true_labels': all_true_labels,
        'probabilities': all_probabilities
    }

    return results

# ============================================================================
# MAIN PIPELINE
# ============================================================================

def main():
    """
    Main pipeline for Optimal TVI model.
    """
    print("="*70)
    print("🏆 TVI-READ: Optimal TVI Framework")
    print("="*70)
    print("\n🎯 Best Performing Model: 85.71% accuracy")
    print("   Innovation: Ablation-guided component selection")
    print("   Components: Only beneficial TVI features")
    print("   Model: Random Forest with deeper trees")

    # Load data
    df = load_data('extracted_features_real_etdd70.csv')

    # Create optimal TVI features
    agg_df = create_optimal_tvi(
        df,
        BASELINE_FEATURES,
        OPTIMAL_TVI_COMPONENTS,
        TVI_WEIGHTS
    )

    # Prepare feature matrix
    feature_cols = (
        [f for f in BASELINE_FEATURES if f in agg_df.columns] +
        [f for f in OPTIMAL_TVI_COMPONENTS if f in agg_df.columns] +
        ['tvi_score', 'weighted_tvi_score']
    )

    X = agg_df[feature_cols].fillna(0).values
    y = agg_df['label'].values

    print(f"\n📊 Feature matrix: {X.shape[0]} subjects × {X.shape[1]} features")

    # Train and evaluate
    results = train_and_evaluate(X, y, MODEL_PARAMS, CV_FOLDS)

    # Print summary
    print("\n" + "="*70)
    print("✅ TRAINING COMPLETE")
    print("="*70)
    print(f"\n🎯 Key Results:")
    print(f"   • Overall Accuracy:    {results['accuracy_mean']*100:.2f}%")
    print(f"   • High-Conf Accuracy:  {results['high_conf_accuracy']*100:.1f}% (on {results['high_conf_percentage']:.1f}% samples)")
    print(f"   • Sensitivity:         {results['sensitivity']*100:.1f}%")
    print(f"   • Specificity:         {results['specificity']*100:.1f}%")

    print("\n💡 Key Innovation:")
    print("   Removed harmful components: cv_fixation_duration, speed_irregularity")
    print("   Kept beneficial components: entropy, autocorrelation, cv_intervals, meta")
    print("   Result: Higher accuracy through scientific component selection!")

    print("\n" + "="*70)

    return results

# ============================================================================
# ENTRY POINT
# ============================================================================

if __name__ == "__main__":
    results = main()

🏆 TVI-READ: Optimal TVI Framework

🎯 Best Performing Model: 85.71% accuracy
   Innovation: Ablation-guided component selection
   Components: Only beneficial TVI features
   Model: Random Forest with deeper trees
📊 Loading data...
✅ Loaded 210 recordings from 70 subjects

🔬 Creating Optimal TVI features...
   Available features: 6 baseline + 4 TVI
   ✅ Created regular and weighted TVI scores

📊 Feature matrix: 70 subjects × 12 features

🤖 Training model with 10-fold cross-validation...

📊 Results:
   Overall Accuracy:     0.8571 ± 0.1107 (85.71%)
   High-Confidence:      0.9412 (94.1%) on 48.6% samples

   Clinical Metrics:
   Sensitivity:          0.8000 (80.0%)
   Specificity:          0.9143 (91.4%)
   PPV:                  0.9032 (90.3%)
   NPV:                  0.8205 (82.1%)

✅ TRAINING COMPLETE

🎯 Key Results:
   • Overall Accuracy:    85.71%
   • High-Conf Accuracy:  94.1% (on 48.6% samples)
   • Sensitivity:         80.0%
   • Specificity:         91.4%

💡 Key Innovation:
   R

In [ ]:
# ЯЧЕЙКА: ПРОДВИНУТЫЕ МЕТОДЫ ДЛЯ ДОСТИЖЕНИЯ 90%+

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score, LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif, RFE, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Предполагаем что agg_df уже создан из предыдущей ячейки
# Если нет - нужно запустить предыдущий код

print("="*80)
print("🚀 ПРОДВИНУТЫЕ МЕТОДЫ ДЛЯ 90%+ ТОЧНОСТИ")
print("="*80)

# Подготовка данных
feature_cols_final = [c for c in agg_df.columns if c not in ['subject_id', 'label']]
X = agg_df[feature_cols_final].fillna(0).values
y = (agg_df['label'] == 'dyslexic').astype(int).values

print(f"\n📊 Исходные данные: {X.shape[0]} субъектов × {X.shape[1]} признаков")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ============================================================================
# МЕТОД 1: LEAVE-ONE-OUT CROSS-VALIDATION
# Авторы часто используют LOOCV для маленьких датасетов
# ============================================================================

print("\n" + "="*80)
print("📊 МЕТОД 1: Leave-One-Out Cross-Validation")
print("="*80)

loo = LeaveOneOut()

models_loocv = {
    'SVM (RBF)': SVC(kernel='rbf', C=1.0, gamma='scale'),
    'SVM (RBF, C=0.5)': SVC(kernel='rbf', C=0.5, gamma='scale'),
    'SVM (RBF, C=2)': SVC(kernel='rbf', C=2.0, gamma='scale'),
    'SVM (Linear)': SVC(kernel='linear', C=1.0),
    'KNN (k=3)': KNeighborsClassifier(n_neighbors=3),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'KNN (k=7)': KNeighborsClassifier(n_neighbors=7),
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0),
}

best_loocv = 0
for name, model in models_loocv.items():
    scores = cross_val_score(model, X_scaled, y, cv=loo, scoring='accuracy')
    acc = scores.mean()
    marker = "🏆" if acc > best_loocv else "  "
    if acc > best_loocv:
        best_loocv = acc
    print(f"{marker} {name:25s}: {acc*100:.2f}%")

# ============================================================================
# МЕТОД 2: FEATURE SELECTION + МОДЕЛЬ
# ============================================================================

print("\n" + "="*80)
print("📊 МЕТОД 2: Отбор признаков + SVM")
print("="*80)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Сначала отбираем лучшие признаки используя f_classif
selector = SelectKBest(f_classif, k='all')
selector.fit(X_scaled, y)
scores_features = selector.scores_
indices = np.argsort(scores_features)[::-1]

print("\nТоп-15 признаков по F-score:")
for i in range(15):
    print(f"   {i+1}. {feature_cols_final[indices[i]]}: {scores_features[indices[i]]:.2f}")

# Пробуем разное количество признаков с разными моделями
print("\n📊 Точность (LOOCV) с разным количеством признаков:")
print("-"*60)

best_k = 0
best_acc = 0
best_model_name = ""

for k in [5, 7, 10, 12, 15, 20, 25, 30]:
    if k > len(feature_cols_final):
        continue

    X_selected = X_scaled[:, indices[:k]]

    for model_name, model in [
        ('SVM', SVC(kernel='rbf', C=1.0, gamma='scale')),
        ('KNN-5', KNeighborsClassifier(n_neighbors=5)),
        ('LR', LogisticRegression(max_iter=1000)),
    ]:
        scores = cross_val_score(model, X_selected, y, cv=loo, scoring='accuracy')
        acc = scores.mean()

        if acc > best_acc:
            best_acc = acc
            best_k = k
            best_model_name = model_name
            print(f"🏆 k={k:2d}, {model_name:6s}: {acc*100:.2f}%")

print(f"\n✅ Лучший результат: {best_model_name} с k={best_k} признаками: {best_acc*100:.2f}%")

# ============================================================================
# МЕТОД 3: PCA + КЛАССИФИКАЦИЯ
# ============================================================================

print("\n" + "="*80)
print("📊 МЕТОД 3: PCA + Классификация")
print("="*80)

best_pca_acc = 0
best_n_components = 0

for n_comp in [5, 10, 15, 20, 25, 30]:
    if n_comp > min(X_scaled.shape):
        continue

    pca = PCA(n_components=n_comp)
    X_pca = pca.fit_transform(X_scaled)

    model = SVC(kernel='rbf', C=1.0, gamma='scale')
    scores = cross_val_score(model, X_pca, y, cv=loo, scoring='accuracy')
    acc = scores.mean()

    marker = "🏆" if acc > best_pca_acc else "  "
    if acc > best_pca_acc:
        best_pca_acc = acc
        best_n_components = n_comp

    print(f"{marker} PCA({n_comp:2d}) + SVM: {acc*100:.2f}%")

# ============================================================================
# МЕТОД 4: MAJORITY VOTING ПО ЗАДАНИЯМ (как INSIGHT)
# ============================================================================

print("\n" + "="*80)
print("📊 МЕТОД 4: Majority Voting по заданиям (INSIGHT подход)")
print("="*80)

# Загружаем исходные данные по заданиям
# df из предыдущей ячейки содержит записи по заданиям

# Создаём отдельные модели для каждого задания
tasks = df['task'].unique()
print(f"Задания: {tasks}")

# Для каждого задания создаём отдельный датафрейм
task_predictions = {}

for task in tasks:
    task_df = df[df['task'] == task].copy()
    task_features = [c for c in task_df.columns if c not in ['subject_id', 'task', 'label']]

    X_task = task_df[task_features].fillna(0).values
    y_task = (task_df['label'] == 'dyslexic').astype(int).values
    subject_ids = task_df['subject_id'].values

    # Масштабирование
    scaler_task = StandardScaler()
    X_task_scaled = scaler_task.fit_transform(X_task)

    # Feature selection для задания
    selector = SelectKBest(f_classif, k=min(15, X_task_scaled.shape[1]))
    X_task_selected = selector.fit_transform(X_task_scaled, y_task)

    # LOOCV для получения предсказаний
    model = SVC(kernel='rbf', C=1.0, gamma='scale')

    predictions = []
    for train_idx, test_idx in loo.split(X_task_selected):
        model.fit(X_task_selected[train_idx], y_task[train_idx])
        pred = model.predict(X_task_selected[test_idx])
        predictions.append((subject_ids[test_idx[0]], pred[0]))

    task_predictions[task] = dict(predictions)

    # Точность для задания
    task_acc = sum(1 for sid, pred in predictions if pred == (labels_dict[sid] == 'dyslexic')) / len(predictions)
    print(f"   {task}: {task_acc*100:.2f}%")

# Majority voting
print("\n📊 Majority Voting результат:")
correct = 0
total = 0

for subject_id in df['subject_id'].unique():
    votes = []
    for task in tasks:
        if subject_id in task_predictions[task]:
            votes.append(task_predictions[task][subject_id])

    if len(votes) >= 2:
        final_pred = 1 if sum(votes) > len(votes) / 2 else 0
        true_label = 1 if labels_dict[subject_id] == 'dyslexic' else 0

        if final_pred == true_label:
            correct += 1
        total += 1

majority_acc = correct / total
print(f"🏆 Majority Voting Accuracy: {majority_acc*100:.2f}%")

# ============================================================================
# МЕТОД 5: HYPERPARAMETER TUNING
# ============================================================================

print("\n" + "="*80)
print("📊 МЕТОД 5: Подбор гиперпараметров SVM")
print("="*80)

# Используем лучший набор признаков
X_best = X_scaled[:, indices[:best_k]]

best_params_acc = 0
best_params = {}

for C in [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    for gamma in ['scale', 'auto', 0.01, 0.1, 1.0]:
        model = SVC(kernel='rbf', C=C, gamma=gamma)
        scores = cross_val_score(model, X_best, y, cv=loo, scoring='accuracy')
        acc = scores.mean()

        if acc > best_params_acc:
            best_params_acc = acc
            best_params = {'C': C, 'gamma': gamma}
            print(f"🏆 C={C}, gamma={gamma}: {acc*100:.2f}%")

print(f"\n✅ Лучшие параметры: {best_params}")
print(f"✅ Лучшая точность: {best_params_acc*100:.2f}%")

# ============================================================================
# МЕТОД 6: ENSEMBLE ИЗ ЛУЧШИХ МОДЕЛЕЙ
# ============================================================================

print("\n" + "="*80)
print("📊 МЕТОД 6: Ансамбль лучших моделей")
print("="*80)

from sklearn.ensemble import VotingClassifier

# Используем лучшие признаки
X_best = X_scaled[:, indices[:best_k]]

# Создаём ансамбль из лучших моделей
ensemble = VotingClassifier(
    estimators=[
        ('svm1', SVC(kernel='rbf', C=best_params['C'], gamma=best_params['gamma'], probability=True)),
        ('svm2', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)),
        ('knn', KNeighborsClassifier(n_neighbors=5)),
        ('lr', LogisticRegression(max_iter=1000)),
    ],
    voting='soft'
)

scores = cross_val_score(ensemble, X_best, y, cv=loo, scoring='accuracy')
print(f"🏆 Ensemble Accuracy (LOOCV): {scores.mean()*100:.2f}%")

# ============================================================================
# ФИНАЛЬНЫЙ РЕЗУЛЬТАТ
# ============================================================================

print("\n" + "="*80)
print("📊 ИТОГОВЫЕ РЕЗУЛЬТАТЫ")
print("="*80)

results = [
    ("Baseline (10-fold CV)", 85.71),
    ("LOOCV + SVM", best_loocv * 100),
    ("Feature Selection + SVM", best_acc * 100),
    ("PCA + SVM", best_pca_acc * 100),
    ("Majority Voting", majority_acc * 100),
    ("Tuned SVM", best_params_acc * 100),
]

results.sort(key=lambda x: x[1], reverse=True)

print("\nРейтинг методов:")
for i, (name, acc) in enumerate(results, 1):
    marker = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "  "
    print(f"{marker} {i}. {name}: {acc:.2f}%")

print("\n" + "="*80)

🚀 ПРОДВИНУТЫЕ МЕТОДЫ ДЛЯ 90%+ ТОЧНОСТИ

📊 Исходные данные: 70 субъектов × 231 признаков

📊 МЕТОД 1: Leave-One-Out Cross-Validation
🏆 SVM (RBF)                : 85.71%
   SVM (RBF, C=0.5)         : 85.71%
   SVM (RBF, C=2)           : 85.71%
   SVM (Linear)             : 84.29%
   KNN (k=3)                : 85.71%
   KNN (k=5)                : 85.71%
   KNN (k=7)                : 85.71%
   Logistic Regression      : 84.29%

📊 МЕТОД 2: Отбор признаков + SVM

Топ-15 признаков по F-score:
   1. metric_n_between_line_regress_trial_T4Me: nan
   2. metric_n_between_line_regress_trial_T5Ps: nan
   3. metric_n_between_line_regress_trial_max: nan
   4. metric_n_between_line_regress_trial_min: nan
   5. metric_n_between_line_regress_trial_std: nan
   6. metric_n_between_line_regress_trial_mean: nan
   7. metric_n_between_line_regress_trial_T1Sy: nan
   8. metric_mean_fix_dur_aoi_max: 61.72
   9. metric_mean_fix_dur_trial_max: 57.97
   10. metric_dwell_time_first_visit_aoi_mean: 53.85
   11. metri

In [ ]:
"""
🎯 Extract Real Demo Examples - FINAL VERSION
Combines feature extraction + model training + example extraction
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import json
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("="*90)
print("🚀 EXTRACTING REAL DEMO EXAMPLES FROM ETDD70")
print("="*90)

# ============================================================================
# LOAD PRE-EXTRACTED FEATURES
# ============================================================================

print("\n📊 STEP 1: Loading pre-extracted features...")

try:
    df = pd.read_csv('extracted_features_real_etdd70.csv')
    print(f"✅ Loaded {len(df)} recordings from {df['subject_id'].nunique()} subjects")
except FileNotFoundError:
    print("❌ File not found: extracted_features_real_etdd70.csv")
    print("   Please run the feature extraction code first!")
    exit(1)

# Convert labels
if df['label'].dtype == 'object':
    label_mapping = {'dyslexic': 1, 'non-dyslexic': 0, 'dyslexia': 1, 'control': 0}
    df['label'] = df['label'].map(label_mapping)

# ============================================================================
# CREATE OPTIMAL TVI FEATURES
# ============================================================================

print("\n📊 STEP 2: Creating Optimal TVI features...")

BASELINE_FEATURES = [
    'num_fixations', 'mean_fixation_duration', 'median_fixation_duration',
    'total_reading_time', 'mean_fixation_x', 'std_fixation_y'
]

OPTIMAL_TVI = [
    'entropy_fixation_duration', 'autocorrelation',
    'cv_inter_fixation_intervals', 'meta_variability'
]

TVI_WEIGHTS = {
    'entropy_fixation_duration': 2.86,
    'autocorrelation': 2.86,
    'cv_inter_fixation_intervals': 1.43,
    'meta_variability': 0.50
}

# Aggregate by subject
available_baseline = [f for f in BASELINE_FEATURES if f in df.columns]
available_tvi = [f for f in OPTIMAL_TVI if f in df.columns]

all_features = available_baseline + available_tvi
agg_dict = {f: 'mean' for f in all_features}
agg_dict['label'] = 'first'

agg_df = df.groupby('subject_id').agg(agg_dict).reset_index()

# Create TVI scores
agg_df['tvi_score'] = agg_df[available_tvi].mean(axis=1)

weighted_sum = sum(agg_df[comp] * TVI_WEIGHTS[comp] for comp in available_tvi)
weight_total = sum(TVI_WEIGHTS[comp] for comp in available_tvi)
agg_df['weighted_tvi_score'] = weighted_sum / weight_total

print(f"✅ Features ready: {len(agg_df)} subjects")

# ============================================================================
# TRAIN MODEL WITH CV
# ============================================================================

print("\n🤖 STEP 3: Training Optimal TVI model...")

feature_cols = available_baseline + available_tvi + ['tvi_score', 'weighted_tvi_score']
X = agg_df[feature_cols].fillna(0).values
y = agg_df['label'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
model = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42)

all_predictions = []
all_probabilities = []
all_true_labels = []
all_indices = []

for train_idx, test_idx in skf.split(X_scaled, y):
    model.fit(X_scaled[train_idx], y[train_idx])
    y_pred = model.predict(X_scaled[test_idx])
    y_proba = model.predict_proba(X_scaled[test_idx])

    all_predictions.extend(y_pred)
    all_probabilities.extend(np.max(y_proba, axis=1))
    all_true_labels.extend(y[test_idx])
    all_indices.extend(test_idx)

all_predictions = np.array(all_predictions)
all_probabilities = np.array(all_probabilities)
all_true_labels = np.array(all_true_labels)
all_indices = np.array(all_indices)

accuracy = accuracy_score(all_true_labels, all_predictions)
print(f"✅ Model trained! Accuracy: {accuracy:.2%}")

# ============================================================================
# EXTRACT 5 DIVERSE EXAMPLES
# ============================================================================

print("\n" + "="*90)
print("📦 STEP 4: Extracting diverse demo examples")
print("="*90)

examples = []

# Strategy: diverse examples
dyslexia_high = np.where((all_predictions == 1) & (all_true_labels == 1) & (all_probabilities > 0.9))[0]
control_high = np.where((all_predictions == 0) & (all_true_labels == 0) & (all_probabilities > 0.9))[0]
medium_conf = np.where((all_probabilities > 0.7) & (all_probabilities < 0.85))[0]

selected = []
if len(dyslexia_high) > 0: selected.append(('dyslexia_high_1', dyslexia_high[0]))
if len(control_high) > 0: selected.append(('control_high_1', control_high[0]))
if len(dyslexia_high) > 1: selected.append(('dyslexia_high_2', dyslexia_high[1]))
if len(control_high) > 1: selected.append(('control_high_2', control_high[1]))
if len(medium_conf) > 0: selected.append(('medium_conf', medium_conf[0]))

# Extract examples
for ex_type, cv_idx in selected[:5]:
    # Get original dataframe index
    orig_idx = all_indices[cv_idx]
    subject_id = int(agg_df.iloc[orig_idx]['subject_id'])

    # Baseline
    baseline = {
        'num_fixations': int(agg_df.iloc[orig_idx].get('num_fixations', 0)),
        'mean_fixation_duration': int(agg_df.iloc[orig_idx].get('mean_fixation_duration', 0)),
        'median_fixation_duration': int(agg_df.iloc[orig_idx].get('median_fixation_duration', 0)),
        'total_reading_time': int(agg_df.iloc[orig_idx].get('total_reading_time', 0)),
        'mean_fixation_x': int(agg_df.iloc[orig_idx].get('mean_fixation_x', 0)),
        'std_fixation_y': int(agg_df.iloc[orig_idx].get('std_fixation_y', 0))
    }

    # TVI
    tvi_components = {
        'entropy_fixation_duration': float(agg_df.iloc[orig_idx].get('entropy_fixation_duration', 0)),
        'autocorrelation': float(agg_df.iloc[orig_idx].get('autocorrelation', 0)),
        'cv_inter_fixation_intervals': float(agg_df.iloc[orig_idx].get('cv_inter_fixation_intervals', 0)),
        'meta_variability': float(agg_df.iloc[orig_idx].get('meta_variability', 0))
    }

    tvi_scores = {
        'components': tvi_components,
        'weightedTVI': float(agg_df.iloc[orig_idx].get('weighted_tvi_score', 0)),
        'regularTVI': float(agg_df.iloc[orig_idx].get('tvi_score', 0))
    }

    # Prediction
    pred_label = 'dyslexia' if all_predictions[cv_idx] == 1 else 'control'
    true_label = 'dyslexia' if all_true_labels[cv_idx] == 1 else 'control'
    confidence = float(all_probabilities[cv_idx])
    conf_level = 'high' if confidence > 0.9 else 'medium' if confidence > 0.7 else 'low'

    # Explanation
    factors = []
    e = tvi_components['entropy_fixation_duration']
    a = tvi_components['autocorrelation']
    c = tvi_components['cv_inter_fixation_intervals']

    if e > 2.5:
        factors.append(f"High entropy ({e:.2f} > 2.5) indicates unpredictable reading patterns")
    else:
        factors.append(f"Normal entropy ({e:.2f} < 2.5) shows predictable reading patterns")

    if abs(a) < 0.3:
        factors.append(f"Low autocorrelation ({abs(a):.2f} < 0.3) indicates irregular sequential patterns")
    else:
        factors.append(f"Good autocorrelation ({abs(a):.2f} > 0.3) shows regular sequential patterns")

    if c > 0.4:
        factors.append(f"High variability in inter-fixation intervals ({c:.2f} > 0.4)")
    else:
        factors.append(f"Low variability in inter-fixation intervals ({c:.2f} < 0.4)")

    recommendation = (
        "Strong indicators detected. Professional assessment recommended." if confidence > 0.9 and pred_label == 'dyslexia'
        else "Reading patterns appear typical. No indicators of dyslexia detected." if confidence > 0.9 and pred_label == 'control'
        else "Mixed indicators. Professional evaluation recommended for comprehensive assessment."
    )

    example = {
        'id': len(examples) + 1,
        'name': f"Subject {subject_id} ({true_label.title()})",
        'description': f"Real data from ETDD70 dataset",
        'actualLabel': true_label,
        'baseline': baseline,
        'tvi': tvi_scores,
        'prediction': {
            'prediction': pred_label,
            'confidence': confidence,
            'confidenceLevel': conf_level,
            'explanation': {
                'mainFactors': factors,
                'recommendation': recommendation
            }
        }
    }

    examples.append(example)

    print(f"\n✅ Example {len(examples)}: Subject {subject_id} ({true_label})")
    print(f"   Type: {ex_type}")
    print(f"   Prediction: {pred_label} | Confidence: {confidence:.1%}")
    print(f"   TVI: {tvi_scores['weightedTVI']:.3f}")

# ============================================================================
# PRINT JAVASCRIPT CODE
# ============================================================================

print("\n" + "="*90)
print("📄 COPY THIS JAVASCRIPT CODE TO: src/data/demoExamples.js")
print("="*90)
print("\n// Real examples from ETDD70 dataset with Optimal TVI model predictions")
print("export const demoExamples = [")
for i, ex in enumerate(examples):
    print("  " + json.dumps(ex, indent=2).replace("\n", "\n  ") + ("," if i < len(examples)-1 else ""))
print("];")
print("\nexport const getDemoExample = (id) => demoExamples.find(ex => ex.id === id);")
print("export const getAllDemoExamples = () => demoExamples;")

print("\n" + "="*90)
print("✅ DONE! Copy the JavaScript code above to your React app")
print("="*90)

🚀 EXTRACTING REAL DEMO EXAMPLES FROM ETDD70

📊 STEP 1: Loading pre-extracted features...
✅ Loaded 210 recordings from 70 subjects

📊 STEP 2: Creating Optimal TVI features...
✅ Features ready: 70 subjects

🤖 STEP 3: Training Optimal TVI model...
✅ Model trained! Accuracy: 85.71%

📦 STEP 4: Extracting diverse demo examples

✅ Example 1: Subject 1038 (dyslexia)
   Type: dyslexia_high_1
   Prediction: dyslexia | Confidence: 92.0%
   TVI: 1.011

✅ Example 2: Subject 1145 (control)
   Type: control_high_1
   Prediction: control | Confidence: 91.5%
   TVI: 1.089

✅ Example 3: Subject 1166 (dyslexia)
   Type: dyslexia_high_2
   Prediction: dyslexia | Confidence: 98.0%
   TVI: 1.015

✅ Example 4: Subject 1255 (control)
   Type: control_high_2
   Prediction: control | Confidence: 92.5%
   TVI: 1.120

✅ Example 5: Subject 1235 (control)
   Type: medium_conf
   Prediction: control | Confidence: 84.0%
   TVI: 0.909

📄 COPY THIS JAVASCRIPT CODE TO: src/data/demoExamples.js

// Real examples from ETD

In [ ]:
# ЯЧЕЙКА: Установка Flask
!pip install flask flask-cors pyngrok -q

In [ ]:
"""
🚀 TVI-Read API Server
Flask API для интеграции модели с React приложением
"""

from flask import Flask, request, jsonify
from flask_cors import CORS
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

app = Flask(__name__)
CORS(app)  # Разрешить запросы от React

# ============================================================================
# ЗАГРУЗКА МОДЕЛИ ПРИ СТАРТЕ
# ============================================================================

print("🚀 Загрузка модели...")

# Параметры модели
BASELINE_FEATURES = [
    'num_fixations',
    'mean_fixation_duration',
    'median_fixation_duration',
    'total_reading_time',
    'mean_fixation_x',
    'std_fixation_y'
]

OPTIMAL_TVI = [
    'entropy_fixation_duration',
    'autocorrelation',
    'cv_inter_fixation_intervals',
    'meta_variability'
]

TVI_WEIGHTS = {
    'entropy_fixation_duration': 2.86,
    'autocorrelation': 2.86,
    'cv_inter_fixation_intervals': 1.43,
    'meta_variability': 0.50
}

# Загружаем обученную модель (или создаем новую)
# ВАРИАНТ 1: Если у вас есть сохраненная модель
# model = pickle.load(open('optimal_tvi_model.pkl', 'rb'))
# scaler = pickle.load(open('scaler.pkl', 'rb'))

# ВАРИАНТ 2: Обучим модель при старте
def train_model():
    """Обучает модель при старте сервера"""
    print("   Загрузка данных...")

    try:
        df = pd.read_csv('extracted_features_real_etdd70.csv')
    except FileNotFoundError:
        print("❌ Файл extracted_features_real_etdd70.csv не найден!")
        return None, None

    # Конвертируем метки
    if df['label'].dtype == 'object':
        label_mapping = {'dyslexic': 1, 'non-dyslexic': 0, 'dyslexia': 1, 'control': 0}
        df['label'] = df['label'].map(label_mapping)

    # Агрегация по субъектам
    available_baseline = [f for f in BASELINE_FEATURES if f in df.columns]
    available_tvi = [f for f in OPTIMAL_TVI if f in df.columns]

    all_features = available_baseline + available_tvi
    agg_dict = {f: 'mean' for f in all_features}
    agg_dict['label'] = 'first'

    agg_df = df.groupby('subject_id').agg(agg_dict).reset_index()

    # TVI scores
    agg_df['tvi_score'] = agg_df[available_tvi].mean(axis=1)
    weighted_sum = sum(agg_df[comp] * TVI_WEIGHTS[comp] for comp in available_tvi)
    weight_total = sum(TVI_WEIGHTS[comp] for comp in available_tvi)
    agg_df['weighted_tvi_score'] = weighted_sum / weight_total

    # Подготовка данных
    feature_cols = available_baseline + available_tvi + ['tvi_score', 'weighted_tvi_score']
    X = agg_df[feature_cols].fillna(0).values
    y = agg_df['label'].values

    # Обучение
    print("   Обучение модели...")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_scaled, y)

    print("✅ Модель обучена!")

    return model, scaler

# Обучаем модель
MODEL, SCALER = train_model()

if MODEL is None:
    print("❌ Не удалось загрузить модель! API не будет работать.")

# ============================================================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ДЛЯ РАСЧЕТА TVI
# ============================================================================

def calculate_tvi_from_fixations(fixations_data):
    """
    Рассчитывает TVI компоненты из данных фиксаций

    fixations_data: list of dicts с ключами: x, y, duration, startTime
    """
    if not fixations_data or len(fixations_data) < 2:
        return None

    durations = np.array([f['duration'] for f in fixations_data])

    # Базовые признаки
    features = {
        'num_fixations': len(fixations_data),
        'mean_fixation_duration': float(durations.mean()),
        'median_fixation_duration': float(np.median(durations)),
        'total_reading_time': float(fixations_data[-1]['startTime'] + fixations_data[-1]['duration'] - fixations_data[0]['startTime']),
        'mean_fixation_x': float(np.mean([f['x'] for f in fixations_data])),
        'std_fixation_y': float(np.std([f['y'] for f in fixations_data]))
    }

    # TVI компоненты

    # 1. Entropy
    hist, _ = np.histogram(durations, bins=10)
    prob = hist / (hist.sum() + 1e-6)
    prob = prob[prob > 0]
    features['entropy_fixation_duration'] = float(-np.sum(prob * np.log2(prob + 1e-6)))

    # 2. CV inter-fixation intervals
    if len(fixations_data) > 1:
        intervals = []
        for i in range(1, len(fixations_data)):
            interval = fixations_data[i]['startTime'] - (fixations_data[i-1]['startTime'] + fixations_data[i-1]['duration'])
            if interval > 0:
                intervals.append(interval)

        if len(intervals) > 0:
            intervals_arr = np.array(intervals)
            features['cv_inter_fixation_intervals'] = float(intervals_arr.std() / (intervals_arr.mean() + 1e-6))
        else:
            features['cv_inter_fixation_intervals'] = 0.0
    else:
        features['cv_inter_fixation_intervals'] = 0.0

    # 3. Autocorrelation
    if len(durations) > 2:
        durations_norm = (durations - durations.mean()) / (durations.std() + 1e-6)
        autocorr = np.correlate(durations_norm[:-1], durations_norm[1:], mode='valid')
        features['autocorrelation'] = float(autocorr[0] / len(durations_norm) if len(autocorr) > 0 else 0)
    else:
        features['autocorrelation'] = 0.0

    # 4. Meta-variability
    if len(durations) > 10:
        window_size = len(durations) // 3
        window_stds = []
        for i in range(len(durations) - window_size):
            window = durations[i:i+window_size]
            window_stds.append(window.std())
        window_stds_arr = np.array(window_stds)
        features['meta_variability'] = float(window_stds_arr.std() / (window_stds_arr.mean() + 1e-6))
    else:
        features['meta_variability'] = 0.0

    # TVI scores
    tvi_components = [
        features['entropy_fixation_duration'],
        features['autocorrelation'],
        features['cv_inter_fixation_intervals'],
        features['meta_variability']
    ]
    features['tvi_score'] = float(np.mean(tvi_components))

    # Weighted TVI
    weighted_sum = (
        TVI_WEIGHTS['entropy_fixation_duration'] * features['entropy_fixation_duration'] +
        TVI_WEIGHTS['autocorrelation'] * abs(features['autocorrelation']) +
        TVI_WEIGHTS['cv_inter_fixation_intervals'] * features['cv_inter_fixation_intervals'] +
        TVI_WEIGHTS['meta_variability'] * features['meta_variability']
    )
    weight_total = sum(TVI_WEIGHTS.values())
    features['weighted_tvi_score'] = float(weighted_sum / weight_total)

    return features

# ============================================================================
# API ENDPOINTS
# ============================================================================

@app.route('/health', methods=['GET'])
def health():
    """Проверка что API работает"""
    return jsonify({
        'status': 'ok',
        'model_loaded': MODEL is not None
    })

@app.route('/predict', methods=['POST'])
def predict():
    """
    Главный endpoint для предсказания

    Ожидает JSON:
    {
        "fixations": [
            {"x": 100, "y": 200, "duration": 250, "startTime": 1000},
            ...
        ]
    }

    Возвращает:
    {
        "prediction": "dyslexia" | "control",
        "confidence": 0.92,
        "tvi_components": {...},
        "baseline_features": {...}
    }
    """

    if MODEL is None:
        return jsonify({'error': 'Model not loaded'}), 500

    try:
        data = request.json
        fixations = data.get('fixations', [])

        if not fixations or len(fixations) < 2:
            return jsonify({'error': 'Insufficient fixation data'}), 400

        # Вычисляем признаки
        features = calculate_tvi_from_fixations(fixations)

        if features is None:
            return jsonify({'error': 'Failed to calculate features'}), 400

        # Подготовка для модели
        feature_order = (
            BASELINE_FEATURES +
            OPTIMAL_TVI +
            ['tvi_score', 'weighted_tvi_score']
        )

        X = np.array([[features.get(f, 0) for f in feature_order]])
        X_scaled = SCALER.transform(X)

        # Предсказание
        prediction_label = int(MODEL.predict(X_scaled)[0])
        prediction_proba = MODEL.predict_proba(X_scaled)[0]
        confidence = float(max(prediction_proba))

        # Формируем ответ
        result = {
            'prediction': 'dyslexia' if prediction_label == 1 else 'control',
            'confidence': confidence,
            'confidenceLevel': 'high' if confidence > 0.9 else 'medium' if confidence > 0.7 else 'low',
            'baseline': {
                'num_fixations': features['num_fixations'],
                'mean_fixation_duration': features['mean_fixation_duration'],
                'median_fixation_duration': features['median_fixation_duration'],
                'total_reading_time': features['total_reading_time'],
                'mean_fixation_x': features['mean_fixation_x'],
                'std_fixation_y': features['std_fixation_y']
            },
            'tvi': {
                'components': {
                    'entropy_fixation_duration': features['entropy_fixation_duration'],
                    'autocorrelation': features['autocorrelation'],
                    'cv_inter_fixation_intervals': features['cv_inter_fixation_intervals'],
                    'meta_variability': features['meta_variability']
                },
                'weightedTVI': features['weighted_tvi_score'],
                'regularTVI': features['tvi_score']
            }
        }

        # Добавляем объяснение
        factors = []
        e = features['entropy_fixation_duration']
        a = features['autocorrelation']
        c = features['cv_inter_fixation_intervals']

        if e > 2.5:
            factors.append(f"High entropy ({e:.2f} > 2.5) indicates unpredictable reading patterns")
        else:
            factors.append(f"Normal entropy ({e:.2f} < 2.5) shows predictable reading patterns")

        if abs(a) < 0.3:
            factors.append(f"Low autocorrelation ({abs(a):.2f} < 0.3) indicates irregular sequential patterns")
        else:
            factors.append(f"Good autocorrelation ({abs(a):.2f} > 0.3) shows regular sequential patterns")

        if c > 0.4:
            factors.append(f"High variability in inter-fixation intervals ({c:.2f} > 0.4)")
        else:
            factors.append(f"Low variability in inter-fixation intervals ({c:.2f} < 0.4)")

        recommendation = (
            "Strong indicators detected. Professional assessment recommended."
            if confidence > 0.9 and result['prediction'] == 'dyslexia'
            else "Reading patterns appear typical. No indicators of dyslexia detected."
            if confidence > 0.9 and result['prediction'] == 'control'
            else "Mixed indicators. Professional evaluation recommended for comprehensive assessment."
        )

        result['explanation'] = {
            'mainFactors': factors,
            'recommendation': recommendation
        }

        return jsonify(result)

    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

# ============================================================================
# ЗАПУСК СЕРВЕРА
# ============================================================================

if __name__ == '__main__':
    print("\n" + "="*70)
    print("🚀 TVI-READ API SERVER")
    print("="*70)
    print("\n✅ API запущен на http://localhost:5000")
    print("\nEndpoints:")
    print("  GET  /health  - Проверка статуса")
    print("  POST /predict - Предсказание по фиксациям")
    print("\n" + "="*70 + "\n")

    app.run(host='0.0.0.0', port=5000, debug=True)

🚀 Загрузка модели...
   Загрузка данных...
   Обучение модели...
✅ Модель обучена!

🚀 TVI-READ API SERVER

✅ API запущен на http://localhost:5000

Endpoints:
  GET  /health  - Проверка статуса
  POST /predict - Предсказание по фиксациям


 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)
